# Statistical Analysis — Zero-Shot LLM Complaint Classification
**Paper:** *Zero-Shot Consumer Financial Complaint Classification Using Open-Source Large Language Models on a High-Performance Computing Cluster*  
**Authors:** Muhammet Tan, Halit Bakır — Sivas University of Science and Technology

This notebook reproduces all statistical tests and numerical results reported in the paper.
Running every cell top-to-bottom regenerates every metric, McNemar p-value, and per-class F1
score directly from the raw prediction CSV files.

**Requirements:** `pandas`, `scipy`, `scikit-learn`
```
pip install pandas scipy scikit-learn
```

In [1]:
import csv
import os
import glob
import re

import numpy as np
import pandas as pd
from scipy.stats import binom
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    classification_report,
)

pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

## 1. Constants & Helpers

In [2]:
CATEGORIES = [
    "retail_banking", "credit_reporting", "credit_card",
    "mortgages_and_loans", "debt_collection",
]

REASONING_MODELS = {"deepseek-r1_14b", "deepseek-r1_8b", "qwen3_8b", "gpt-oss_20b"}

BALANCED_DIR   = "balanced_preds"
IMBALANCED_DIR = "imbalanced_preds"

# Prefix patterns match the naming convention used in classify.py / SLURM jobs
BALANCED_PREFIX   = r"^preds_(?:pilot_)?"
IMBALANCED_PREFIX = r"^preds_imb_"


def clean_name(path, prefix_pat):
    base = os.path.basename(path)
    name = re.sub(prefix_pat, "", base)
    return re.sub(r"\.csv$", "", name)


def load_preds(path):
    """Load a prediction CSV, normalise labels, mark off-label predictions."""
    df = pd.read_csv(path)
    df["true_label"]      = df["true_label"].str.lower().str.strip()
    df["predicted_label"] = df["predicted_label"].str.lower().str.strip()
    df.loc[~df["predicted_label"].isin(CATEGORIES), "predicted_label"] = "off_label"
    return df


def compute_metrics(df):
    """Aggregate metrics identical to analyze.py."""
    y_true = df["true_label"]
    y_pred = df["predicted_label"]
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, labels=CATEGORIES, average="weighted", zero_division=0)
    rec  = recall_score(   y_true, y_pred, labels=CATEGORIES, average="weighted", zero_division=0)
    f1w  = f1_score(       y_true, y_pred, labels=CATEGORIES, average="weighted", zero_division=0)
    f1m  = f1_score(       y_true, y_pred, labels=CATEGORIES, average="macro",    zero_division=0)
    lat  = df["latency_seconds"].astype(float)
    return {
        "accuracy":       round(acc,  4),
        "precision":      round(prec, 4),
        "recall":         round(rec,  4),
        "f1_weighted":    round(f1w,  4),
        "f1_macro":       round(f1m,  4),
        "mean_latency_s": round(lat.mean(), 2),
        "off_label":      int((y_pred == "off_label").sum()),
        "n":              len(df),
    }


def load_experiment(src_dir, prefix_pat):
    """Load all CSVs in a directory and return a summary DataFrame + raw dict."""
    files = sorted(glob.glob(os.path.join(src_dir, "*.csv")))
    records = []
    raw = {}
    for path in files:
        model = clean_name(path, prefix_pat)
        df = load_preds(path)
        m  = compute_metrics(df)
        m["model"] = model
        records.append(m)
        raw[model] = df
    cols = ["model", "accuracy", "precision", "recall",
            "f1_weighted", "f1_macro", "mean_latency_s", "off_label", "n"]
    summary = pd.DataFrame(records)[cols].sort_values("accuracy", ascending=False).reset_index(drop=True)
    return summary, raw

print("Helpers loaded.")

Helpers loaded.


---
## 2. Experiment A — Balanced Evaluation (200 samples / class, N = 1 000)

In [3]:
sum_a, raw_a = load_experiment(BALANCED_DIR, BALANCED_PREFIX)

# Add model type label
sum_a["type"] = sum_a["model"].apply(lambda m: "Reasoning" if m in REASONING_MODELS else "Standard")

print("Experiment A — Balanced Results")
print("=" * 90)
display_cols = ["model", "type", "accuracy", "precision", "recall",
                "f1_weighted", "f1_macro", "mean_latency_s", "off_label"]
print(sum_a[display_cols].to_string(index=False))

Experiment A — Balanced Results
          model      type  accuracy  precision  recall  f1_weighted  f1_macro  mean_latency_s  off_label
    gpt-oss_20b Reasoning    0.7450     0.7635  0.7450       0.7451    0.7451          2.5900          0
deepseek-r1_14b Reasoning    0.7160     0.7330  0.7160       0.7162    0.7162          5.9200          0
       phi4_14b  Standard    0.7090     0.7357  0.7090       0.7095    0.7095          0.5500          1
       qwen3_8b Reasoning    0.7070     0.7390  0.7070       0.7066    0.7066          3.8900          0
      gemma2_9b  Standard    0.6760     0.7166  0.6760       0.6728    0.6728          0.6600          1
 deepseek-r1_8b Reasoning    0.6750     0.7091  0.6750       0.6736    0.6736          4.6400          1
     mistral_7b  Standard    0.5960     0.6756  0.5960       0.5842    0.5842          0.2400         54
    llama3.1_8b  Standard    0.5870     0.6966  0.5870       0.5958    0.5958          0.5700         57


### Notes
- In the balanced setting, **macro-F1 == weighted F1** because all five classes have equal support (200 samples each).
- Off-label predictions are counted as errors for all metrics.

---
## 3. Experiment B — Imbalanced Evaluation (natural distribution, N = 1 000)

In [4]:
sum_b, raw_b = load_experiment(IMBALANCED_DIR, IMBALANCED_PREFIX)

sum_b["type"] = sum_b["model"].apply(lambda m: "Reasoning" if m in REASONING_MODELS else "Standard")

print("Experiment B — Imbalanced Results")
print("=" * 90)
print(sum_b[display_cols].to_string(index=False))

Experiment B — Imbalanced Results
          model      type  accuracy  precision  recall  f1_weighted  f1_macro  mean_latency_s  off_label
    gpt-oss_20b Reasoning    0.8040     0.8058  0.8040       0.8042    0.7237          2.3900          0
       qwen3_8b Reasoning    0.7980     0.7985  0.7980       0.7950    0.7025          3.6000          0
       phi4_14b  Standard    0.7980     0.8034  0.7980       0.7987    0.7004          0.5400          1
deepseek-r1_14b Reasoning    0.7900     0.8016  0.7900       0.7936    0.7017          5.7200          0
 deepseek-r1_8b Reasoning    0.7710     0.7765  0.7710       0.7681    0.6632          4.4100          4
      gemma2_9b  Standard    0.7620     0.7845  0.7620       0.7617    0.6605          0.6600          0
    llama3.1_8b  Standard    0.6870     0.7824  0.6870       0.7092    0.5555          0.5600         49
     mistral_7b  Standard    0.6860     0.7774  0.6860       0.7033    0.5360          0.2300         36


### Class distribution in Experiment B

In [5]:
# Use any model's true labels — they are identical across files
sample_model_b = list(raw_b.values())[0]
dist = sample_model_b["true_label"].value_counts(normalize=True).round(4) * 100
print("True class distribution (Experiment B):")
print(dist.to_string())

True class distribution (Experiment B):
true_label
credit_reporting      63.6000
debt_collection       14.1000
mortgages_and_loans    7.7000
credit_card            7.5000
retail_banking         7.1000


---
## 4. McNemar Tests — Experiment A

We apply **exact (binomial) McNemar tests** on paired correctness outcomes.
Given two binary vectors $c_1, c_2 \in \{0,1\}^{1000}$:
- $b$ = count where model 1 correct, model 2 wrong
- $c$ = count where model 1 wrong, model 2 correct
- Two-sided p-value: $p = 2 \cdot \text{Binomial-CDF}(\min(b,c),\; b+c,\; 0.5)$

Bonferroni-adjusted significance threshold for 4 comparisons against the best model: $\alpha/4 = 0.05/4 = 0.0125$.

In [6]:
def correct_vector(df):
    return (df["true_label"] == df["predicted_label"]).astype(int).values


def exact_mcnemar(v1, v2):
    """Two-sided exact McNemar test on paired binary correctness vectors."""
    assert len(v1) == len(v2)
    b = int(((v1 == 1) & (v2 == 0)).sum())  # v1 correct, v2 wrong
    c = int(((v1 == 0) & (v2 == 1)).sum())  # v1 wrong,   v2 correct
    n = b + c
    if n == 0:
        return b, c, 1.0
    p = min(2 * binom.cdf(min(b, c), n, 0.5), 1.0)
    return b, c, p


vecs_a = {model: correct_vector(df) for model, df in raw_a.items()}

ALPHA_BONFERRONI = 0.05 / 4  # 4 comparisons vs best model

best = "gpt-oss_20b"

print(f"Best model: {best}")
print(f"Bonferroni-adjusted alpha (4 comparisons): {ALPHA_BONFERRONI:.4f}")
print()

# --- Significant comparisons (best vs. next-best models) ---
sig_pairs = ["deepseek-r1_14b", "phi4_14b", "qwen3_8b", "gemma2_9b"]

print("Significant comparisons (best model vs. others):")
print(f"  {'Model A':22s}  {'Model B':22s}  {'b':>4}  {'c':>4}  {'p-value':>10}  sig?")
print("  " + "-" * 75)
for other in sig_pairs:
    b_cnt, c_cnt, p = exact_mcnemar(vecs_a[best], vecs_a[other])
    sig = "YES" if p < ALPHA_BONFERRONI else "no"
    print(f"  {best:22s}  {other:22s}  {b_cnt:4d}  {c_cnt:4d}  {p:10.4f}  {sig}")

print()
# --- Non-significant pairs (adjacent top models) ---
ns_pairs = [
    ("phi4_14b",        "qwen3_8b"),
    ("deepseek-r1_14b", "phi4_14b"),
    ("deepseek-r1_14b", "qwen3_8b"),
]

print("Non-significant pairs (adjacent top models):")
print(f"  {'Model A':22s}  {'Model B':22s}  {'b':>4}  {'c':>4}  {'p-value':>10}  sig?")
print("  " + "-" * 75)
for ma, mb in ns_pairs:
    b_cnt, c_cnt, p = exact_mcnemar(vecs_a[ma], vecs_a[mb])
    sig = "YES" if p < 0.05 else "no"
    print(f"  {ma:22s}  {mb:22s}  {b_cnt:4d}  {c_cnt:4d}  {p:10.4f}  {sig}")

Best model: gpt-oss_20b
Bonferroni-adjusted alpha (4 comparisons): 0.0125

Significant comparisons (best model vs. others):
  Model A                 Model B                    b     c     p-value  sig?
  ---------------------------------------------------------------------------
  gpt-oss_20b             deepseek-r1_14b           75    46      0.0106  YES
  gpt-oss_20b             phi4_14b                  88    52      0.0030  YES
  gpt-oss_20b             qwen3_8b                  81    43      0.0008  YES
  gpt-oss_20b             gemma2_9b                115    46      0.0000  YES

Non-significant pairs (adjacent top models):
  Model A                 Model B                    b     c     p-value  sig?
  ---------------------------------------------------------------------------
  phi4_14b                qwen3_8b                  71    69      0.9327  no
  deepseek-r1_14b         phi4_14b                  76    69      0.6184  no
  deepseek-r1_14b         qwen3_8b                

**Interpretation:**
- `gpt-oss_20b` is statistically superior to all four compared models at the Bonferroni-adjusted threshold.
- Adjacent top-three models (`deepseek-r1_14b`, `phi4_14b`, `qwen3_8b`) are **not** significantly different from each other — their rank order should be interpreted cautiously.

---
## 5. Per-Class F1 for Leading Models

In [7]:
def per_class_f1(df, model_name, exp_label):
    """Compute per-class F1 for a single model's prediction DataFrame."""
    y_true = df["true_label"]
    y_pred = df["predicted_label"]
    scores = f1_score(y_true, y_pred, labels=CATEGORIES, average=None, zero_division=0)
    row = {"Exp": exp_label, "Model": model_name}
    row.update(dict(zip(CATEGORIES, [round(s, 4) for s in scores])))
    return row


leading_a = ["gpt-oss_20b", "phi4_14b", "qwen3_8b"]
leading_b = ["gpt-oss_20b", "phi4_14b", "qwen3_8b"]

rows = []
for m in leading_a:
    rows.append(per_class_f1(raw_a[m], m, "A"))
for m in leading_b:
    rows.append(per_class_f1(raw_b[m], m, "B"))

per_class_df = pd.DataFrame(rows)

col_rename = {
    "retail_banking":      "Retail",
    "credit_reporting":    "CreditRep.",
    "credit_card":         "Card",
    "mortgages_and_loans": "Mortg.",
    "debt_collection":     "Debt",
}
per_class_df.rename(columns=col_rename, inplace=True)

print("Per-Class F1 for Leading Models (Table 5 in the paper)")
print("=" * 75)
print(per_class_df.to_string(index=False))

Per-Class F1 for Leading Models (Table 5 in the paper)
Exp       Model  Retail  CreditRep.   Card  Mortg.   Debt
  A gpt-oss_20b  0.8098      0.7195 0.7154  0.7902 0.6906
  A    phi4_14b  0.7259      0.7054 0.6998  0.7457 0.6705
  A    qwen3_8b  0.7700      0.6870 0.6613  0.7740 0.6409
  B gpt-oss_20b  0.7771      0.8810 0.6316  0.7179 0.6109
  B    phi4_14b  0.7006      0.8831 0.6182  0.6624 0.6378
  B    qwen3_8b  0.7595      0.8801 0.5750  0.7006 0.5975


---
## 6. Per-Class Recall for gpt-oss_20b (Experiment A)

The paper states: *gpt-oss:20b reaches recall 0.885 on credit_reporting but only 0.625 on debt_collection*

In [8]:
from sklearn.metrics import classification_report

df_best = raw_a["gpt-oss_20b"]
print("gpt-oss_20b — Experiment A — Classification Report")
print(classification_report(
    df_best["true_label"],
    df_best["predicted_label"],
    labels=CATEGORIES,
    zero_division=0,
))

gpt-oss_20b — Experiment A — Classification Report
                     precision    recall  f1-score   support

     retail_banking       0.79      0.83      0.81       200
   credit_reporting       0.61      0.89      0.72       200
        credit_card       0.78      0.66      0.72       200
mortgages_and_loans       0.87      0.72      0.79       200
    debt_collection       0.77      0.62      0.69       200

           accuracy                           0.74      1000
          macro avg       0.76      0.74      0.75      1000
       weighted avg       0.76      0.74      0.75      1000



---
## 7. Off-Label Count Verification

In [9]:
print(f"{'Model':22s}  {'Exp A OL':>8}  {'Exp B OL':>8}")
print("-" * 44)
for m in sorted(raw_a.keys()):
    ol_a = int((raw_a[m]["predicted_label"] == "off_label").sum())
    # match imbalanced name — same model name after prefix strip
    ol_b = int((raw_b.get(m, raw_b.get(m, pd.DataFrame({"predicted_label":[]})))["predicted_label"] == "off_label").sum()) if m in raw_b else "N/A"
    print(f"{m:22s}  {ol_a:8d}  {str(ol_b):>8}")

Model                   Exp A OL  Exp B OL
--------------------------------------------
deepseek-r1_14b                0         0
deepseek-r1_8b                 1         4
gemma2_9b                      1         0
gpt-oss_20b                    0         0
llama3.1_8b                   57        49
mistral_7b                    54        36
phi4_14b                       1         1
qwen3_8b                       0         0


---
## 8. Comparison with Reference Paper (Table 4)

Reference results sourced from Roumeliotis et al. (2025), Table 2.

In [10]:
reference = pd.DataFrame([
    {"model": "o1 (paid API)",                "source": "Reference", "accuracy": 0.777, "f1": 0.7764, "cost_per_1k": "$24.24"},
    {"model": "GPT-4.5 (paid API)",            "source": "Reference", "accuracy": 0.774, "f1": 0.7763, "cost_per_1k": "$12.32"},
    {"model": "Claude 3.7 Sonnet (paid API)",   "source": "Reference", "accuracy": 0.765, "f1": 0.7626, "cost_per_1k": "$0.64"},
    {"model": "Gemini 2.0 Flash (paid API)",    "source": "Reference", "accuracy": 0.758, "f1": 0.7586, "cost_per_1k": "$0.04"},
    {"model": "GPT-4o (paid API)",              "source": "Reference", "accuracy": 0.736, "f1": 0.7358, "cost_per_1k": "$0.45"},
])

ours = sum_a[["model", "accuracy", "f1_weighted"]].copy()
ours.rename(columns={"f1_weighted": "f1"}, inplace=True)
ours["source"] = "Ours (local)"
ours["cost_per_1k"] = "$0.00"

combined = pd.concat([reference, ours], ignore_index=True).sort_values("accuracy", ascending=False)
print("Combined Comparison (Table 4 extension)")
print("=" * 72)
print(combined[["model", "source", "accuracy", "f1", "cost_per_1k"]].to_string(index=False))

# Key claim
our_best_f1  = sum_a.loc[sum_a["model"]=="gpt-oss_20b", "f1_weighted"].values[0]
gpt4o_f1     = 0.7358
print(f"\ngpt-oss:20b F1 ({our_best_f1:.4f}) > GPT-4o F1 ({gpt4o_f1:.4f}): {our_best_f1 > gpt4o_f1}")

Combined Comparison (Table 4 extension)
                       model       source  accuracy     f1 cost_per_1k
               o1 (paid API)    Reference    0.7770 0.7764      $24.24
          GPT-4.5 (paid API)    Reference    0.7740 0.7763      $12.32
Claude 3.7 Sonnet (paid API)    Reference    0.7650 0.7626       $0.64
 Gemini 2.0 Flash (paid API)    Reference    0.7580 0.7586       $0.04
                 gpt-oss_20b Ours (local)    0.7450 0.7451       $0.00
           GPT-4o (paid API)    Reference    0.7360 0.7358       $0.45
             deepseek-r1_14b Ours (local)    0.7160 0.7162       $0.00
                    phi4_14b Ours (local)    0.7090 0.7095       $0.00
                    qwen3_8b Ours (local)    0.7070 0.7066       $0.00
                   gemma2_9b Ours (local)    0.6760 0.6728       $0.00
              deepseek-r1_8b Ours (local)    0.6750 0.6736       $0.00
                  mistral_7b Ours (local)    0.5960 0.5842       $0.00
                 llama3.1_8b Ours (lo

---
## 9. Summary of Results Reported in Paper

Quick sanity-check that all key numbers in the paper match the CSV-derived values.

In [11]:
checks = [
    # (description, computed_value, paper_value)
    ("gpt-oss_20b  Exp-A accuracy",    sum_a.loc[sum_a.model=="gpt-oss_20b",  "accuracy"    ].values[0], 0.7450),
    ("gpt-oss_20b  Exp-A F1",          sum_a.loc[sum_a.model=="gpt-oss_20b",  "f1_weighted" ].values[0], 0.7451),
    ("deepseek-r1_14b Exp-A F1",       sum_a.loc[sum_a.model=="deepseek-r1_14b","f1_weighted"].values[0], 0.7162),
    ("phi4_14b  Exp-A F1",             sum_a.loc[sum_a.model=="phi4_14b",     "f1_weighted" ].values[0], 0.7095),
    ("qwen3_8b  Exp-A F1",             sum_a.loc[sum_a.model=="qwen3_8b",     "f1_weighted" ].values[0], 0.7066),
    ("mistral_7b  Exp-A F1",           sum_a.loc[sum_a.model=="mistral_7b",   "f1_weighted" ].values[0], 0.5842),
    ("llama3.1_8b Exp-A F1",           sum_a.loc[sum_a.model=="llama3.1_8b",  "f1_weighted" ].values[0], 0.5958),
    ("gpt-oss_20b  Exp-B accuracy",    sum_b.loc[sum_b.model=="gpt-oss_20b",  "accuracy"    ].values[0], 0.8040),
    ("gpt-oss_20b  Exp-B F1",          sum_b.loc[sum_b.model=="gpt-oss_20b",  "f1_weighted" ].values[0], 0.8042),
    ("gpt-oss_20b  Exp-B macro-F1",    sum_b.loc[sum_b.model=="gpt-oss_20b",  "f1_macro"    ].values[0], 0.7237),
    ("llama3.1_8b Exp-B weighted-F1",  sum_b.loc[sum_b.model=="llama3.1_8b",  "f1_weighted" ].values[0], 0.7092),
    ("llama3.1_8b Exp-B macro-F1",     sum_b.loc[sum_b.model=="llama3.1_8b",  "f1_macro"    ].values[0], 0.5555),
]

all_ok = True
print(f"{'Description':45s}  {'Computed':>10}  {'Paper':>10}  {'Match?':>6}")
print("-" * 80)
for desc, computed, paper in checks:
    match = abs(computed - paper) < 0.00015   # allow 1 ULP at 4dp
    all_ok = all_ok and match
    flag = "OK" if match else "MISMATCH"
    print(f"{desc:45s}  {computed:10.4f}  {paper:10.4f}  {flag:>6}")

print()
print("All checks passed:" if all_ok else "WARNING: some values do not match the paper.")

Description                                      Computed       Paper  Match?
--------------------------------------------------------------------------------
gpt-oss_20b  Exp-A accuracy                        0.7450      0.7450      OK
gpt-oss_20b  Exp-A F1                              0.7451      0.7451      OK
deepseek-r1_14b Exp-A F1                           0.7162      0.7162      OK
phi4_14b  Exp-A F1                                 0.7095      0.7095      OK
qwen3_8b  Exp-A F1                                 0.7066      0.7066      OK
mistral_7b  Exp-A F1                               0.5842      0.5842      OK
llama3.1_8b Exp-A F1                               0.5958      0.5958      OK
gpt-oss_20b  Exp-B accuracy                        0.8040      0.8040      OK
gpt-oss_20b  Exp-B F1                              0.8042      0.8042      OK
gpt-oss_20b  Exp-B macro-F1                        0.7237      0.7237      OK
llama3.1_8b Exp-B weighted-F1                      0.7092    

---
## 10. Reproducibility Notes

| Item | Value |
|------|-------|
| Balanced CSV dir | `balanced_preds/` |
| Imbalanced CSV dir | `imbalanced_preds/` |
| Classification script | `classify.py` |
| Analysis script | `analyze.py` |
| SLURM job file | `run_ollama.slurm` |
| Ollama version | v0.30.6 |
| GPU | NVIDIA Tesla V100-SXM2-16GB |
| Cluster | TRUBA ARF, akya-cuda partition |
| Random seed (sampling) | 42 |
| Temperature | 0 (all models) |
| Quantisation | q4 (Ollama default) |

All prediction CSVs (raw model responses + parsed labels + per-sample latency) are committed to the repository at:  
https://github.com/muhammettan28/ConsumerComplaint